# SAR Oil Spill Detection — Training Notebook

This notebook trains the U-Net model using preprocessed patches from Google Drive.

## Before running:
1. Make sure you have already run `colab_preprocessing.ipynb` at least once
2. Go to **Runtime → Change runtime type → T4 GPU**
3. Run all cells from top to bottom

## What this notebook does:
- Loads preprocessed patches from Drive (skips preprocessing entirely)
- Trains U-Net on GPU
- Plots training curves
- Saves best model and results to Drive
- Commits results to GitHub

---
## Cell 1 — Verify GPU
Always run this first. If it says CPU, go to Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

---
## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

def gb(x): return x / (1024 ** 3)
total, used, free = shutil.disk_usage('/content')
print(f'Colab local disk — total: {gb(total):.1f} GB | free: {gb(free):.1f} GB')

---
## Cell 3 — Locate Preprocessed Patches on Drive

This cell finds the patches saved by `colab_preprocessing.ipynb`.
They are stored at `MyDrive/oil_spill_results/patches/`.

In [ ]:
import os, json

RESULTS_DIR   = '/content/drive/MyDrive/oil_spill_results'
PATCHES_DRIVE = f'{RESULTS_DIR}/patches'
STATS_DRIVE   = f'{RESULTS_DIR}/train_stats.json'

if not os.path.exists(PATCHES_DRIVE):
    raise FileNotFoundError(
        f'Patches not found at {PATCHES_DRIVE}\n'
        'Please run colab_preprocessing.ipynb first.'
    )

print(f'Patches found at: {PATCHES_DRIVE}')
for split in ['train', 'val', 'test']:
    n = len([f for f in os.listdir(f'{PATCHES_DRIVE}/{split}') if f.endswith('_img.npy')])
    print(f'  {split:<6}: {n} patches')

with open(STATS_DRIVE) as f:
    stats = json.load(f)
print(f'\nNormalization stats:')
print(f'  mean: {stats["mean"]}')
print(f'  std : {stats["std"]}')

---
## Cell 4 — Copy Patches to Colab Local Disk

Training reads each patch hundreds of times across epochs.
Reading from local SSD is significantly faster than reading from Drive on every batch.
This copy takes 5-10 minutes but saves much more time during training.

In [ ]:
import shutil, os

LOCAL_PATCHES = '/content/data/patches'
LOCAL_STATS   = '/content/data/train_stats.json'

os.makedirs('/content/data', exist_ok=True)

if os.path.exists(LOCAL_PATCHES):
    print('Patches already on local disk — skipping copy')
else:
    print('Copying patches from Drive to local disk...')
    shutil.copytree(PATCHES_DRIVE, LOCAL_PATCHES)
    print('Done')

if not os.path.exists(LOCAL_STATS):
    shutil.copy(STATS_DRIVE, LOCAL_STATS)

result = !du -sh /content/data/patches
print(f'Local patches size: {result[0].split()[0]}')
print('Ready for training')

---
## Cell 5 — Clone GitHub Repository

In [ ]:
import os

# Replace with your actual GitHub repo URL
GITHUB_URL = 'https://github.com/TigranBoyakhchyan/GeoSpill-AI'
REPO_NAME  = 'GeoSpill-AI'

if os.path.exists(f'/content/{REPO_NAME}'):
    print('Repo already exists — pulling latest changes...')
    %cd /content/{REPO_NAME}
    !git pull origin main
else:
    print('Cloning repository...')
    !git clone {GITHUB_URL}
    %cd /content/{REPO_NAME}

# Symlink /content/data into the repo so train.py finds patches at data/patches
if not os.path.exists('data'):
    os.symlink('/content/data', 'data')
    print('Symlinked /content/data -> data/')

print(f'Working directory: {os.getcwd()}')

---
## Cell 6 — Install Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio

import segmentation_models_pytorch as smp
import albumentations as A
print(f'segmentation-models-pytorch : {smp.__version__}')
print(f'albumentations              : {A.__version__}')
print('All packages ready')

---
## Cell 7 — Train the Model

Expected time: **15-30 minutes** on T4 GPU for 50 epochs.

You will see a live progress bar per epoch and a summary line after each one.
Training stops automatically if val IoU stops improving for 10 consecutive epochs.

In [ ]:
!python src/train.py \
    --epochs 50 \
    --batch_size 16 \
    --lr 1e-4 \
    --model_type smp

---
## Cell 8 — Plot Training Curves

Key things to look for:
- **Both losses decreasing** — model is learning correctly
- **Val IoU plateauing while Train IoU keeps rising** — overfitting
- **Both metrics flat from epoch 1** — learning rate too high or data problem

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('checkpoints/training_log.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(log['epoch'], log['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val Loss',   color='orange')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(log['epoch'], log['train_iou'], label='Train IoU', color='steelblue')
axes[1].plot(log['epoch'], log['val_iou'],   label='Val IoU',   color='orange')
axes[1].set_title('IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

best = log.loc[log['val_iou'].idxmax()]
print(f'Best epoch   : {int(best["epoch"])}')
print(f'Best val IoU : {best["val_iou"]:.4f}')
print(f'Best val Dice: {best["val_dice"]:.4f}')

---
## Cell 9 — Save Results to Drive

**Always run this before closing the session.**
Colab deletes everything in `/content/` when the session ends.

In [ ]:
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/oil_spill_results'
os.makedirs(SAVE_DIR, exist_ok=True)

files_to_save = {
    'checkpoints/best_model.pth':      f'{SAVE_DIR}/best_model.pth',
    'checkpoints/training_log.csv':    f'{SAVE_DIR}/training_log.csv',
    'checkpoints/training_curves.png': f'{SAVE_DIR}/training_curves.png',
}

for src, dst in files_to_save.items():
    shutil.copy(src, dst)
    print(f'Saved: {src} -> Drive')

print(f'\nAll results saved to: {SAVE_DIR}')

---
## Cell 10 — Commit Results to GitHub

Commits the training log and curves to GitHub.
The `.pth` model file is NOT committed (too large, already in .gitignore).

**Replace email and name below with yours.**

In [ ]:
import pandas as pd

log      = pd.read_csv('checkpoints/training_log.csv')
best     = log.loc[log['val_iou'].idxmax()]
best_iou  = best['val_iou']
best_dice = best['val_dice']

!git config user.email "you@example.com"
!git config user.name  "Your Name"

!git add checkpoints/training_log.csv
!git add checkpoints/training_curves.png
!git add data/train_stats.json

commit_msg = f'Colab training: val IoU={best_iou:.4f}, Dice={best_dice:.4f} ({len(log)} epochs)'
!git commit -m "{commit_msg}"
!git push origin main

print(f'Pushed: {commit_msg}')